In [ ]:
# ==============================================================================
# 12. Model Training (XGBoost / LightGBM)
# ==============================================================================
#
# 목적:
#   1. XGBoost, LightGBM 모델 학습
#   2. Validation 성능 평가
#   3. Test 성능 평가
#   4. 모델 저장
#
# 입력: train.csv, val.csv, test.csv, scale_pos_weights.json
# 출력: 학습된 모델 (.pkl), 성능 리포트
# ==============================================================================

import pandas as pd
import numpy as np
import json
import os
import pickle
from datetime import datetime

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../models'

# 모델 저장 디렉토리 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=== 12. Model Training 시작 ===")

# --- 데이터 로드 ---
print("\nStep 1: 데이터 로드")

df_train = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
df_val = pd.read_csv(os.path.join(INPUT_DIR, 'val.csv'))
df_test = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))

print(f"✓ Train: {len(df_train):,} rows")
print(f"✓ Val: {len(df_val):,} rows")
print(f"✓ Test: {len(df_test):,} rows")

# --- 피처/레이블 로드 ---
with open(os.path.join(INPUT_DIR, 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)

with open(os.path.join(INPUT_DIR, 'scale_pos_weights.json'), 'r') as f:
    scale_pos_weights = json.load(f)

label_cols = [col for col in df_train.columns if 'next_' in col]

print(f"✓ 피처: {len(feature_cols)}개")
print(f"✓ 레이블: {len(label_cols)}개")

=== 12. Model Training 시작 ===

Step 1: 데이터 로드
✓ Train: 657,172 rows
✓ Val: 141,844 rows
✓ Test: 142,801 rows
✓ 피처: 66개
✓ 레이블: 12개


In [2]:
# ==============================================================================
# Step 2: 피처/레이블 분리
# ==============================================================================

print("\nStep 2: 피처/레이블 분리")

X_train = df_train[feature_cols]
X_val = df_val[feature_cols]
X_test = df_test[feature_cols]

print(f"✓ X_train shape: {X_train.shape}")
print(f"✓ X_val shape: {X_val.shape}")
print(f"✓ X_test shape: {X_test.shape}")

# 결측 확인
train_missing = X_train.isna().sum().sum()
val_missing = X_val.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

if train_missing + val_missing + test_missing == 0:
    print("✓ 결측 없음 확인")
else:
    print(f"⚠️ 결측 발견: Train={train_missing}, Val={val_missing}, Test={test_missing}")


Step 2: 피처/레이블 분리
✓ X_train shape: (657172, 66)
✓ X_val shape: (141844, 66)
✓ X_test shape: (142801, 66)
✓ 결측 없음 확인


In [3]:
# ==============================================================================
# Step 3: 학습 타겟 선택
# ==============================================================================
#
# 주요 타겟:
#   - death_next_24h: 24시간 내 사망 예측
#   - vent_next_24h: 24시간 내 인공호흡기 시작 예측
#   - pressor_next_24h: 24시간 내 승압제 시작 예측
#   - composite_next_24h: 위 3개 중 하나라도 발생
#
# 여기서는 모든 레이블에 대해 학습하되, 메인 타겟 지정
# ==============================================================================

print("\nStep 3: 학습 타겟 선택")

# 메인 타겟 (우선순위 높은 것들)
main_targets = [
    'death_next_24h',
    'vent_next_24h', 
    'pressor_next_24h',
    'composite_next_24h'
]

# 존재하는 타겟만 필터링
main_targets = [t for t in main_targets if t in label_cols]

print(f"메인 타겟: {main_targets}")


Step 3: 학습 타겟 선택
메인 타겟: ['death_next_24h', 'vent_next_24h', 'pressor_next_24h', 'composite_next_24h']


In [12]:
# ==============================================================================
# Step 4: 모델 학습 함수 정의
# ==============================================================================

def train_and_evaluate(model, model_name, X_train, y_train, X_val, y_val, X_test, y_test, target_name):
    """
    모델 학습 및 평가
    """
    print(f"\n{'='*50}")
    print(f"{model_name} - {target_name}")
    print('='*50)
    
    # 학습
    print("학습 중...")
    
    if model_name == 'XGBoost':
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    else:  # LightGBM
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )
    
    # 예측
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    y_test_pred = model.predict_proba(X_test)[:, 1]
    
    # 평가 지표 계산
    results = {
        'model': model_name,
        'target': target_name,
        'train_auroc': roc_auc_score(y_train, y_train_pred),
        'val_auroc': roc_auc_score(y_val, y_val_pred),
        'test_auroc': roc_auc_score(y_test, y_test_pred),
    }
    
    # AUPRC
    precision, recall, _ = precision_recall_curve(y_test, y_test_pred)
    results['test_auprc'] = auc(recall, precision)
    
    # 최적 threshold에서 F1
    best_f1 = 0
    best_threshold = 0.5
    for threshold in np.arange(0.1, 0.9, 0.05):
        y_pred_binary = (y_test_pred >= threshold).astype(int)
        f1 = f1_score(y_test, y_pred_binary, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    results['test_f1'] = best_f1
    results['best_threshold'] = best_threshold
    
    # 결과 출력
    print(f"\n  Train AUROC: {results['train_auroc']:.4f}")
    print(f"  Val AUROC:   {results['val_auroc']:.4f}")
    print(f"  Test AUROC:  {results['test_auroc']:.4f}")
    print(f"  Test AUPRC:  {results['test_auprc']:.4f}")
    print(f"  Test F1:     {results['test_f1']:.4f} (threshold={results['best_threshold']:.2f})")
    
    # Confusion Matrix
    y_pred_binary = (y_test_pred >= best_threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_binary)
    
    print(f"\n  Confusion Matrix (threshold={best_threshold:.2f}):")
    print(f"    TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"    FN={cm[1,0]:,}  TP={cm[1,1]:,}")
    
    # Overfitting 체크
    overfit_gap = results['train_auroc'] - results['val_auroc']
    if overfit_gap > 0.05:
        print(f"\n  ⚠️ Overfitting 의심: Train-Val gap = {overfit_gap:.4f}")
    
    return results, model, y_test_pred

In [13]:
# ==============================================================================
# Step 5: XGBoost 학습
# ==============================================================================

print("\n" + "="*60)
print("Step 5: XGBoost 학습")
print("="*60)

xgb_results = []
xgb_models = {}
xgb_predictions = {}

for target in main_targets:
    # 레이블 추출
    y_train = df_train[target]
    y_val = df_val[target]
    y_test = df_test[target]
    
    # scale_pos_weight
    spw = scale_pos_weights.get(target, 1.0)
    
    # 모델 정의
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=-1,
        eval_metric='auc',
        early_stopping_rounds=30
    )
    
    # 학습 및 평가
    result, model, y_pred = train_and_evaluate(
        xgb_model, 'XGBoost', 
        X_train, y_train, 
        X_val, y_val, 
        X_test, y_test,
        target
    )
    
    xgb_results.append(result)
    xgb_models[target] = model
    xgb_predictions[target] = y_pred


Step 5: XGBoost 학습

XGBoost - death_next_24h
학습 중...

  Train AUROC: 0.9772
  Val AUROC:   0.8829
  Test AUROC:  0.9275
  Test AUPRC:  0.2694
  Test F1:     0.3464 (threshold=0.85)

  Confusion Matrix (threshold=0.85):
    TN=140,796  FP=934
    FN=651  TP=420

  ⚠️ Overfitting 의심: Train-Val gap = 0.0944

XGBoost - vent_next_24h
학습 중...

  Train AUROC: 0.8948
  Val AUROC:   0.7409
  Test AUROC:  0.7533
  Test AUPRC:  0.1450
  Test F1:     0.2063 (threshold=0.70)

  Confusion Matrix (threshold=0.70):
    TN=134,122  FP=4,965
    FN=2,716  TP=998

  ⚠️ Overfitting 의심: Train-Val gap = 0.1539

XGBoost - pressor_next_24h
학습 중...

  Train AUROC: 0.9273
  Val AUROC:   0.8153
  Test AUROC:  0.8219
  Test AUPRC:  0.0836
  Test F1:     0.1475 (threshold=0.80)

  Confusion Matrix (threshold=0.80):
    TN=138,293  FP=2,676
    FN=1,473  TP=359

  ⚠️ Overfitting 의심: Train-Val gap = 0.1120

XGBoost - composite_next_24h
학습 중...

  Train AUROC: 0.8675
  Val AUROC:   0.7634
  Test AUROC:  0.7859
  Tes

In [14]:
# ==============================================================================
# Step 6: LightGBM 학습
# ==============================================================================

print("\n" + "="*60)
print("Step 6: LightGBM 학습")
print("="*60)

lgb_results = []
lgb_models = {}
lgb_predictions = {}

for target in main_targets:
    # 레이블 추출
    y_train = df_train[target]
    y_val = df_val[target]
    y_test = df_test[target]
    
    # scale_pos_weight
    spw = scale_pos_weights.get(target, 1.0)
    
    # 모델 정의
    lgb_model = LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=-1,
        verbosity=-1  # verbose=-1 → verbosity=-1
    )
    
    # 학습 및 평가
    result, model, y_pred = train_and_evaluate(
        lgb_model, 'LightGBM', 
        X_train, y_train, 
        X_val, y_val, 
        X_test, y_test,
        target
    )
    
    lgb_results.append(result)
    lgb_models[target] = model
    lgb_predictions[target] = y_pred


Step 6: LightGBM 학습

LightGBM - death_next_24h
학습 중...

  Train AUROC: 0.9962
  Val AUROC:   0.8652
  Test AUROC:  0.9224
  Test AUPRC:  0.3103
  Test F1:     0.3219 (threshold=0.85)

  Confusion Matrix (threshold=0.85):
    TN=140,627  FP=1,103
    FN=654  TP=417

  ⚠️ Overfitting 의심: Train-Val gap = 0.1310

LightGBM - vent_next_24h
학습 중...

  Train AUROC: 0.9332
  Val AUROC:   0.7296
  Test AUROC:  0.7433
  Test AUPRC:  0.1405
  Test F1:     0.1977 (threshold=0.75)

  Confusion Matrix (threshold=0.75):
    TN=135,846  FP=3,241
    FN=2,951  TP=763

  ⚠️ Overfitting 의심: Train-Val gap = 0.2036

LightGBM - pressor_next_24h
학습 중...

  Train AUROC: 0.9689
  Val AUROC:   0.7953
  Test AUROC:  0.8067
  Test AUPRC:  0.0726
  Test F1:     0.1356 (threshold=0.80)

  Confusion Matrix (threshold=0.80):
    TN=137,743  FP=3,226
    FN=1,464  TP=368

  ⚠️ Overfitting 의심: Train-Val gap = 0.1737

LightGBM - composite_next_24h
학습 중...

  Train AUROC: 0.9039
  Val AUROC:   0.7579
  Test AUROC:  0.785

In [15]:
# ==============================================================================
# Step 7: 결과 비교
# ==============================================================================

print("\n" + "="*60)
print("Step 7: 결과 비교")
print("="*60)

# 결과 DataFrame
all_results = xgb_results + lgb_results
df_results = pd.DataFrame(all_results)

print("\n=== 전체 결과 ===")
print(df_results[['model', 'target', 'val_auroc', 'test_auroc', 'test_auprc', 'test_f1']].to_string(index=False))

# 모델별 평균 성능
print("\n=== 모델별 평균 성능 ===")
model_avg = df_results.groupby('model')[['test_auroc', 'test_auprc', 'test_f1']].mean()
print(model_avg.round(4))

# 타겟별 최고 성능 모델
print("\n=== 타겟별 최고 성능 모델 ===")
for target in main_targets:
    target_results = df_results[df_results['target'] == target]
    best_idx = target_results['test_auroc'].idxmax()
    best_row = df_results.loc[best_idx]
    print(f"  {target}: {best_row['model']} (AUROC={best_row['test_auroc']:.4f})")


Step 7: 결과 비교

=== 전체 결과 ===
   model             target  val_auroc  test_auroc  test_auprc  test_f1
 XGBoost     death_next_24h   0.882852    0.927484    0.269402 0.346392
 XGBoost      vent_next_24h   0.740854    0.753347    0.145015 0.206262
 XGBoost   pressor_next_24h   0.815285    0.821893    0.083568 0.147524
 XGBoost composite_next_24h   0.763441    0.785876    0.218517 0.267697
LightGBM     death_next_24h   0.865190    0.922403    0.310276 0.321883
LightGBM      vent_next_24h   0.729595    0.743344    0.140542 0.197720
LightGBM   pressor_next_24h   0.795253    0.806667    0.072641 0.135643
LightGBM composite_next_24h   0.757901    0.785442    0.224263 0.269740

=== 모델별 평균 성능 ===
          test_auroc  test_auprc  test_f1
model                                    
LightGBM      0.8145      0.1869   0.2312
XGBoost       0.8221      0.1791   0.2420

=== 타겟별 최고 성능 모델 ===
  death_next_24h: XGBoost (AUROC=0.9275)
  vent_next_24h: XGBoost (AUROC=0.7533)
  pressor_next_24h: XGBoost (AUR

In [17]:
# ==============================================================================
# Step 8: 모델 저장
# ==============================================================================

print("\n" + "="*60)
print("Step 8: 모델 저장")
print("="*60)

# 타임스탬프
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# XGBoost 모델 저장
for target, model in xgb_models.items():
    model_path = os.path.join(OUTPUT_DIR, f'xgb_{target}_{timestamp}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ {model_path}")

# LightGBM 모델 저장
for target, model in lgb_models.items():
    model_path = os.path.join(OUTPUT_DIR, f'lgb_{target}_{timestamp}.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ {model_path}")

# 결과 저장
results_path = os.path.join(OUTPUT_DIR, f'results_{timestamp}.csv')
df_results.to_csv(results_path, index=False)
print(f"✓ {results_path}")

# Test 예측값 저장 (SHAP 분석용)
predictions = {
    'stay_id': df_test['stay_id'].values,
    'observation_hour': df_test['observation_hour'].values,
}
for target in main_targets:
    predictions[f'xgb_{target}_prob'] = xgb_predictions[target]
    predictions[f'lgb_{target}_prob'] = lgb_predictions[target]
    predictions[f'{target}_actual'] = df_test[target].values

df_predictions = pd.DataFrame(predictions)
pred_path = os.path.join(OUTPUT_DIR, f'test_predictions_{timestamp}.csv')
df_predictions.to_csv(pred_path, index=False)
print(f"✓ {pred_path}")

print("\n=== 12. Model Training 완료 ===")


Step 8: 모델 저장
✓ ../models/xgb_death_next_24h_20260107_222330.pkl
✓ ../models/xgb_vent_next_24h_20260107_222330.pkl
✓ ../models/xgb_pressor_next_24h_20260107_222330.pkl
✓ ../models/xgb_composite_next_24h_20260107_222330.pkl
✓ ../models/lgb_death_next_24h_20260107_222330.pkl
✓ ../models/lgb_vent_next_24h_20260107_222330.pkl
✓ ../models/lgb_pressor_next_24h_20260107_222330.pkl
✓ ../models/lgb_composite_next_24h_20260107_222330.pkl
✓ ../models/results_20260107_222330.csv
✓ ../models/test_predictions_20260107_222330.csv

=== 12. Model Training 완료 ===
